# Data Unterstanding - Olist Marketplace

## Zweck
Dieses Notebook dokumentiert Struktur, Beziehungen und Datenqualität der Rohdaten.

Ziel ist es zu prüfen, ob der Datensatz für eine Analyse geeignet ist.
Es werden keine Transformationen, Features oder Business-Interpretationen vorgenommen.

In [6]:
import duckdb
from pathlib import Path

# Verbindung zur DuckDB (in-memory reicht völlig)
con = duckdb.connect()

# Hilfsfunktion für SQL-Abfragen
def sql(q):
    return con.sql(q).df()

# Projektpfad bestimmen (Notebook liegt in /notebooks)
DATA = Path("../data/raw/brazilian-ecommerce")

In [8]:
# Alle CSV-Dateien in DuckDB registrieren
for f in DATA.glob("*.csv"):
    name = f.stem.replace("olist_", "").replace("_dataset", "")
    
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('{f.as_posix()}')
        """
           )
           
# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## 1. Datensatzüberblick

Der Datensatz enthält Transaktionsdaten eines Online-Marketplaces.
Bestellungen verbinden Kunden, Händler, Produkte, Zahlungen, Bewertungen und Lieferinformationen.

Da die Informationen auf mehrere relationale Tabellen verteilt sind, muss zunächst deren Aufbau und Zusammenhang verstanden werden.

## 2. Tabellenstruktur & Spaltenbeschreibung

### Customer/Seller/Geolocation

| Column | Datenniveau | Description |
|------|------|------|
| customer_id | kategorisch (nominal) | Kunden-Order-ID |
| customer_unique_id | kategorisch (nominal) | Kunden-ID |
| customer_zip_code_prefix/seller_zip_code_prefix/geolocation_zip_code_prefix | kategorisch (nominal) | Kunden/Verkäufer/Geolocation Postleitzahl |
| customer_city/seller_city/geolocation_city | kategorisch (nominal) | Kunden/Verkäufer/Geolocation Stadt |
| customer_state/seller_state/geolocation_state | kategorisch (nominal) | Kunden/Verkäufer/Geolocation Bundesstaat |
| seller_id | kategorisch (nominal) | Verkäufer-ID |
| geolocation_lat | numerisch (`float`)  | Breitengrad |
| geolocation_Ing | numerisch (`float`) | Längengrad |

### Products
| Column | Datenniveau | Description |
|------|------|------|
| product_id | kategorisch (nominal) | eindeutige Produkt(Item)-ID |
| product_category_name | kategorisch (nominal) | Produkkategorie in Portugiesisch |
| product_name_lenght | numerisch (`int`) | Länge des Produktnames |
| product_description_lenght | numerisch (`int`) | Länge der Produktbeschreibung|
| product_photos_qty | numerisch (`int`) | Anzahl der Produktfotos|
| product_weight_g | numerisch (`float`) | Produktgewicht in gramm|
| product_length_cm | numerisch (`float`) | Produktlänge in cm|
| product_height_cm | numerisch (`float`) | Produkthöhe in cm|
| product_width_cm | numerisch (`float`) | Produktbreite in cm|


### Orders

Jede Zeile entspricht einer Bestellung.
enthält auch: customer_id

| Column | Datenniveau | Description |
|------|------|------|
| order_id | kategorisch (nominal) | eindeutige Bestell-ID |
| order_status | kategorisch (nominal) | Bestellstatus |
| order_purchase_timestamp | zeitlich (Timestamp) | Kaufzeitpunkt |
| order_approved_at | zeitlich (Datum) | Zeitpunkt der Zahlungsgenehmigung |
| order_delivered_carrier_date | zeitlich (Datum) | Zeipunkt an dem Bestellung an Logistikpartner übergeben wurde |
| order_delivered_customer_date | zeitlich (Datum) | Lieferdatum |
| order_estimated_delivery_date | zeitlich (Datum) | vorraussichtliches Lieferdatum |

### Orders_items

enthält auch: product_id, seller_id, order_id

| Column | Datenniveau | Description |
|------|------|------|
| order_item_id | kategorisch (nominal)| fortlaufende Nummer für Items IN EINER Bestellung |
| shipping_limit_date | zeitlich (Timestamp) | Zeitlimit des Verkäufers für die Übergabe an den Logistikpartner |
| price | numerisch (`float`) | Preis des Produktes(Items) |
| freight_value | numerisch (`float`) | anteilige Versandkosten für dieses Item |


### Orders_Payments
enthält auch: order_id

| Column | Datenniveau | Description |
|------|------|------|
| payment_sequential | ordinal (Reihenfolge(`int`))| Sortiert + nummeriert TEILZAHLUNGEN derselben Bestellung (1-n pro  order_id )! |
| payment_type | kategorisch (nominal)| Zahlungsart |
| payment_installments | kardinal (Anzahl(`int`)) | Ratenzahlung - Anzahl der vom Kunden gewählten Raten |
| payment_value | numerisch (`float`) | Zahlungswert |

### Orders_Reviews
enthält auch: order_id

| Column | Datenniveau | Description |
|------|------|------|
| review_id | kategorisch (nominal)| eindeutige Review-ID |
| review_score | ordinal (Skala(`int`))| Bewertung-Score - Sternebewertung 1-5|
| review_comment_title | nominal (Text(`str`)) | Review Kommentar-Titel |
| review_comment_message | nominal (Text(`str`))| Review Kommentar |
| review_creation_date | zeitlich (Datum)| Datum des Reviewerstellung |
| review_answer_timestamp | zeitlich (Timestamp) | Zeitstempel der Antwort auf die Review |

### Product_Category_Name_Translation
| Column | Datenniveau | Description |
|------|------|------|
| product_category_name_english | kategorisch (nominal(`str`)) | Produkkategorie in Englisch |
| product_category_name | kategorisch (nominal(`str`)) | Produkkategorie in Portugiesisch |


Hier kommen noch kurze Übersichten rein wie head(), info(), describe()

## 3. Datenbeziehungen (Join Logic)

ER-Diagramm um alle Beziehungen zu zeigen. Liefert Wissen für zukünftige JOINs

![Entity Relationship Diagram](../docs/olist_erd.png)

## **4. Datenqualitätsprüfung**

Vor einer Analyse muss geprüft werden, ob die Daten vollständig und logisch konsistent sind.

###  4.1 Missing Values (Nur relevante Spalten)

In [17]:
# Customers, kann jede Bestellung einem Kunden zugeordnet werden? 
sql("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(customer_unique_id IS NULL) AS sum_missing_customer_uid
FROM customers
    """)

,total_rows,sum_missing_customer_uid
0,99441,0.0


In [ ]:
# Orders, prüfen ob Lieferzeit berechnet werden kann
sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(order_purchase_timestamp IS NULL) AS missing_purchase_timestamp,
    SUM(order_delivered_customer_date IS NULL) AS missing_delivered_timestamp,
    SUM(order_estimated_delivery_date IS NULL) AS missing_estimated_timestamp
FROM orders
    """)

,total_rows,missing_purchase_timestamp,missing_delivered_timestamp,missing_estimated_timestamp
0,99441,0.0,2965.0,0.0


In [ ]:
# Order_Items, prüfen ob Artikelpreise und Mengen vollständig sind
sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(price IS NULL) AS missing_price,
    SUM(freight_value IS NULL) AS missing_freight_value
FROM order_items
    """)

,total_rows,missing_price,missing_freight_value
0,112650,0.0,0.0


In [ ]:
# Payments, prüfen ob Zahlungsbeträge vollständig sind
sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(payment_value IS NULL) AS missing_payment_value
FROM order_payments
    """)

,total_rows,missing_payment_value
0,103886,0.0


In [30]:
# Reviews, prüfen ob Bewertungen vollständig sind
sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(review_score IS NULL) AS missing_review_score
FROM order_reviews
    """)

,total_rows,missing_review_score
0,99224,0.0


Die für Umsatz-, Lieferzeit-, Kunden- und Zufriedenheitsanalysen notwendigen Felder sind weitgehend vollständig. 
Fehlende Werte werden bei der Feature Erstellung berücksichtigt.

### 4.2 Duplicates

In [39]:
# Customers, prüfen ob jede Zeile einen eindeutigen customer_id besitzt
sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS distinct_customer_id,
    COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_id
FROM customers
    """)

,total_rows,distinct_customer_id,duplicate_customer_id
0,99441,99441,0


In [44]:
# Customers, prüfen ob mehrere Bestellungen derselben Person zugeordnet werden können (Wiederkäufer)
sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_unique_id) AS distinct_customer_unique_id,
    COUNT(*) - COUNT(DISTINCT customer_unique_id) AS duplicate_customer_unique_id
FROM customers
    """)

,total_rows,distinct_customer_unique_id,duplicate_customer_unique_id
0,99441,96096,3345


In [41]:
#Orders, prüfen ob jede Bestellung nur einmal existiert
sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS distinct_order_id,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_id
FROM orders
    """)

,total_rows,distinct_order_id,duplicate_order_id
0,99441,99441,0


In [42]:
# Sellers, prüfen ob jeder Verkäufer eindeutig identifizierbar ist
sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT seller_id) AS distinct_seller_id,
    COUNT(*) - COUNT(DISTINCT seller_id) AS duplicate_seller_id
FROM sellers
    """)

,total_rows,distinct_seller_id,duplicate_seller_id
0,3095,3095,0


In [43]:
# Products, prüfen ob jeder Artikel eindeutig identifizierbar ist
sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS distinct_product_id,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_id
FROM products
    """)

,total_rows,distinct_product_id,duplicate_product_id
0,32951,32951,0


Die zentralen Entitäten (Bestellungen, Kunden, Verkäufer und Produkte) besitzen eindeutige Schlüssel und können daher zuverlässig aggregiert werden.
Mehrfach vorkommende customer_unique_id repräsentieren Wiederkäufe derselben Person.

### 4.3 Data Type Issues

### 4.4 Logical Validity Checks (Sind einzelne Werte technisch möglich?)
z.B. Lieferung vor Bestellung, negativer Preis, Bewertungen außerhalb der typsichen 1-5 Skala

## 5. **Zeitliche Abdeckung**

Zur Vermeidung falscher Trends muss geprüft werden,
ob der Beobachtungszeitraum vollständig und konsistent ist.


## 6. **Plausibilitätsprüfung von Verteilungen**

Ziel ist es festzustellen, ob die Daten realistisches Kunden- und Bestellverhalten abbilden.
Es werden keine geschäftlichen Schlussfolgerungen gezogen.

- Bewertungsverteiltung realistisch? z.B. Histogramm

- Preisverteilung z.B. Histogramm

- Lieferdauer z.B. Histogramm/Boxplot

- Bestellungen über Zeit z.B. Liniendiagramm 
-- schwankt die Nachfrage?
-- wirkt es wie echte Kundenaktivität?

## 7. **Datenrisiken und Interpretationseinschränkungen**

Der Datensatz bildet reale Geschäftsprozesse ab, enthält jedoch Einschränkungen, die bei späteren Auswertungen berücksichtigt werden müssen.

1. Fehlende Bewertungen
Bewertungen sind optional. Bestellungen ohne Bewertung stellen daher nicht automatisch unzufriedene Kunden dar.

    Konsequenz:
    Die durchschnittliche Bewertung kann nicht direkt als Zufriedenheitsquote interpretiert werden.

2. Mehrere Artikel pro Bestellung
Eine Bestellung kann mehrere Artikel und Händler enthalten.

    Konsequenz:
    Kennzahlen auf Bestellebene können Liefer- oder Performanceeffekte einzelner Händler verzerren.

3. Marketplace - Struktur
Die Plattform bündelt die Angebote vieler unabhängiger Händler.

    Konsequenz:
    Gesamtergebnisse spiegeln Plattform-Performance wider, nicht die Qualität eines einzelnen Anbieters.

## 8. **Fazit**
Struktur, Beziehungen und Datenqualität des Datensatzes wurden untersucht.

Die Daten bilden die wesentlichen Geschäftsprozesse eines Online-Marketplaces ab
und sind grundsätzlich für eine Analyse geeignet.
Allerdings liegen die Informationen verteilt über mehrere relationale Tabellen vor
und müssen zunächst zu einer auswertbaren Struktur zusammengeführt werden.

Für die weitere Analyse sind daher folgende Schritte erforderlich:

- Zusammenführung der Tabellen auf Bestell- bzw. Kundenebene
- Berechnung von Lieferdauer und Verzögerungsindikatoren
- Aggregation von Zahlungs- und Artikelinformationen
- Ableitung kundenbezogener Kennzahlen

Im nächsten Schritt wird ein analysefähiger Datensatz erstellt.